# 📊 Exploratory Data Analysis: Pulmonary Abnormalities (Shenzhen & Montgomery TB Datasets)

> **Datasets Included:**
> 1. **Shenzhen No.3 Hospital Chest X-ray Set:** 662 images (336 TB, 326 Normal)
> 2. **Montgomery County Chest X-ray Set:** 138 images (58 TB, 80 Normal)
> **Source:** Kaggle `kmader/pulmonary-chest-xray-abnormalities`  
> **License:** Public Domain / NIH / NLM (National Library of Medicine)

---

## 🎯 Purpose & Dataset Usage in Graduation Thesis

1. **Adult Tuberculosis (TB) Class Integration:** Used to construct the 4th class (`Tuberculosis`) for the CDSS demo system.
2. **Honest External Validation Strategy:**
   - **Shenzhen Dataset:** Used for **training & validation** of the Tuberculosis class.
   - **Montgomery Dataset:** Held out **100% as an External Test Set**. Model never sees Montgomery images during training, allowing true evaluation of generalization capability across medical centers.


In [ ]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path("../data/raw/pulmonary_abnormalities")
print(f"Data directory exists: {DATA_DIR.exists()}")


## 1. File Scanning & Parsing (Shenzhen vs Montgomery)

- **Shenzhen:** Filename `CHNCXR_{id}_{label}.png` (`0`=Normal, `1`=TB) -> Patient ID `CHN_{id}`
- **Montgomery:** Filename `MCUCXR_{id}_{label}.png` (`0`=Normal, `1`=TB) -> Patient ID `MCU_{id}`


In [ ]:
raw_images = [
    p for p in DATA_DIR.rglob("*.[pP][nN][gG]")
    if "__MACOSX" not in str(p) and not p.name.startswith("._")
]

records = []
for p in raw_images:
    fn = p.name
    if fn.startswith("CHNCXR"):
        parts = p.stem.split("_")
        if len(parts) >= 3:
            label = "tuberculosis" if parts[2] == "1" else "normal"
            records.append({
                "filepath": str(p),
                "source": "shenzhen",
                "label": label,
                "patient_id": f"CHN_{parts[1]}"
            })
    elif fn.startswith("MCUCXR"):
        parts = p.stem.split("_")
        if len(parts) >= 3:
            label = "tuberculosis" if parts[2] == "1" else "normal"
            records.append({
                "filepath": str(p),
                "source": "montgomery",
                "label": label,
                "patient_id": f"MCU_{parts[1]}"
            })

df = pd.DataFrame(records)
print(f"Total scanned images: {len(df)}")
print(df['source'].value_counts())
df.head()


## 2. Source & Label Breakdown Comparison

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.countplot(data=df, x="source", hue="label", palette=["#2b5c8f", "#e74c3c"])
plt.title("Label Counts by Dataset Source (Shenzhen vs Montgomery)", fontsize=14, fontweight='bold')
plt.xlabel("Dataset Source")
plt.ylabel("Number of X-Rays")

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height + 5), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


## 3. High-Resolution Scans Analysis

Shenzhen and Montgomery datasets contain high-resolution adult DICOM-converted PNG scans.


In [ ]:
widths, heights = [], []
for p in df['filepath'].sample(min(500, len(df)), random_state=42):
    with Image.open(p) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)

df_dim = pd.DataFrame({"width": widths, "height": heights})

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_dim, x="width", y="height", color="#8e44ad", alpha=0.7)
plt.title("Resolution Distribution of Adult X-Rays (Width vs Height)", fontsize=14, fontweight='bold')
plt.xlabel("Width (pixels)")
plt.ylabel("Height (pixels)")
plt.show()

print(f"Mean Resolution: {np.mean(widths):.0f} x {np.mean(heights):.0f} pixels")


## 4. Visual Comparison: Normal vs Tuberculosis

Tuberculosis lesions typically manifest in apical/upper lung zones as cavitating lesions, infiltrates, or pleural effusions.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Shenzhen TB vs Normal
for i, label in enumerate(['normal', 'tuberculosis']):
    samples = df[(df['source'] == 'shenzhen') & (df['label'] == label)].sample(2, random_state=42).reset_index()
    for j in range(2):
        ax = axes[i, j]
        img = Image.open(samples.iloc[j]['filepath']).convert('L')
        ax.imshow(img, cmap='gray')
        ax.set_title(f"SHENZHEN | {label.upper()}
Patient: {samples.iloc[j]['patient_id']}", fontsize=9)
        ax.axis('off')

# Montgomery TB vs Normal
for i, label in enumerate(['normal', 'tuberculosis']):
    samples = df[(df['source'] == 'montgomery') & (df['label'] == label)].sample(2, random_state=42).reset_index()
    for j in range(2):
        ax = axes[i, j + 2]
        img = Image.open(samples.iloc[j]['filepath']).convert('L')
        ax.imshow(img, cmap='gray')
        ax.set_title(f"MONTGOMERY | {label.upper()}
Patient: {samples.iloc[j]['patient_id']}", fontsize=9)
        ax.axis('off')

plt.suptitle("Adult Chest X-Rays: Normal vs Tuberculosis (Shenzhen & Montgomery)", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## 📌 Summary & Key Takeaways for Project Execution

| Aspect | Insight / Strategy |
| :--- | :--- |
| **External Validation Isolation** | All Montgomery images (`MCUCXR_*`) are strictly held out for external validation (`external_test.csv`). |
| **Adult Anatomy** | Adult ribcages are larger, vertical, with elongated lung fields compared to pediatric Kermany X-rays. |
| **Resolution Standardization** | High-res scans (~3000x3000) are downsampled to 224x224 for uniform model input. |
